In [1]:
!pip install -qq statsbombpy \
networkx \
python-louvain \
python-igraph \
leidenalg \
infomap \
scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00


In [1]:
from statsbombpy import sb

import os
import numpy as np
import pandas as pd
import networkx as nx
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict

# ============================================================
# COMMUNITY DETECTION
# ============================================================

import community as community_louvain
import igraph as ig
import leidenalg
from infomap import Infomap

# ============================================================
# METRICS
# ============================================================

from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    f1_score
)

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "open-data/data"

MATCHES_FILE = os.path.join(
    DATA_DIR,
    "matches"
)

EVENTS_DIR = os.path.join(
    DATA_DIR,
    "events"
)

LINEUPS_DIR = os.path.join(
    DATA_DIR,
    "lineups"
)

SEMIFINALISTS = [
    "Argentina",
    "France",
    "Croatia",
    "Morocco"
]

MIN_MINUTES = 30

ModuleNotFoundError: No module named 'igraph'

In [4]:
# ============================================================
# LOAD MATCHES USING STATSBOMBPY
# ============================================================

# FIFA World Cup 2022
matches = sb.matches(
    competition_id=43,
    season_id=106
)

team_matches = defaultdict(list)

for _, match in matches.iterrows():

    home = match["home_team"]
    away = match["away_team"]

    if home in SEMIFINALISTS:
        team_matches[home].append(
            match["match_id"]
        )

    if away in SEMIFINALISTS:
        team_matches[away].append(
            match["match_id"]
        )

/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [5]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def load_events(match_id):

    path = os.path.join(
        EVENTS_DIR,
        f"{match_id}.json"
    )

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_lineups(match_id):

    path = os.path.join(
        LINEUPS_DIR,
        f"{match_id}.json"
    )

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [12]:
# ============================================================
# BUILD PASSING NETWORK
# ============================================================

def build_team_network(team_name):

    G = nx.DiGraph()

    minutes_played = defaultdict(float)
    positions = defaultdict(list)

    # ========================================================
    # LOAD LINEUPS
    # ========================================================

    for match_id in team_matches[team_name]:

        lineups = sb.lineups(match_id)

        lineup_df = lineups[team_name]

        for _, row in lineup_df.iterrows():

            player = row["player_name"]

            # Approximation
            minutes_played[player] += 90

            pos_data = row.get("positions")

            if isinstance(pos_data, list) and len(pos_data) > 0:

                first_position = pos_data[0]

                if "position" in first_position:

                    positions[player].append(
                        first_position["position"]
                    )

    # ========================================================
    # FILTER PLAYERS
    # ========================================================

    valid_players = {
        p for p, m in minutes_played.items()
        if m >= 30
    }

    # ========================================================
    # LOAD EVENTS
    # ========================================================

    for match_id in team_matches[team_name]:

        events = sb.events(match_id)

        passes = events[
            (events["type"] == "Pass") &
            (events["team"] == team_name) &
            (events["pass_outcome"].isna())
        ]

        for _, row in passes.iterrows():

            passer = row["player"]
            recipient = row["pass_recipient"]

            if pd.isna(recipient):
                continue

            if passer not in valid_players:
                continue

            if recipient not in valid_players:
                continue

            if G.has_edge(passer, recipient):

                G[passer][recipient]["weight"] += 1

            else:

                G.add_edge(
                    passer,
                    recipient,
                    weight=1
                )

    # ========================================================
    # NODE ATTRIBUTES
    # ========================================================

    for player in valid_players:

        if player not in G.nodes:
            continue

        line = "UNK"

        if len(positions[player]) > 0:

            pos = positions[player][0].lower()

            if "goalkeeper" in pos:

                line = "GK"

            elif (
                "back" in pos or
                "center back" in pos or
                "wing back" in pos
            ):

                line = "DEF"

            elif "midfield" in pos:

                line = "MID"

            else:

                line = "ATT"

        nx.set_node_attributes(
            G,
            {
                player: {
                    "line": line
                }
            }
        )

    return G

In [13]:
# ============================================================
# LOUVAIN
# ============================================================

def run_louvain(G):

    UG = nx.Graph()

    for u, v, d in G.edges(data=True):

        w = d["weight"]

        if UG.has_edge(u, v):
            UG[u][v]["weight"] += w
        else:
            UG.add_edge(u, v, weight=w)

    partition = community_louvain.best_partition(
        UG,
        weight="weight",
        random_state=42
    )

    modularity = community_louvain.modularity(
        partition,
        UG,
        weight="weight"
    )

    return partition, modularity

In [14]:
# ============================================================
# LEIDEN
# ============================================================

def run_leiden(G):

    UG = nx.Graph()

    for u, v, d in G.edges(data=True):

        w = d["weight"]

        if UG.has_edge(u, v):
            UG[u][v]["weight"] += w
        else:
            UG.add_edge(u, v, weight=w)

    nodes = list(UG.nodes())

    edges = list(UG.edges())

    weights = [
        UG[u][v]["weight"]
        for u, v in edges
    ]

    g = ig.Graph()

    g.add_vertices(nodes)
    g.add_edges(edges)

    g.es["weight"] = weights

    partition = leidenalg.find_partition(
        g,
        leidenalg.RBConfigurationVertexPartition,
        weights=weights
    )

    membership = {
        nodes[i]: partition.membership[i]
        for i in range(len(nodes))
    }

    return membership, partition.modularity

In [15]:
# ============================================================
# INFOMAP
# ============================================================

def run_infomap(G):

    im = Infomap("--directed")

    node_to_id = {}
    id_to_node = {}

    for idx, node in enumerate(G.nodes()):

        node_to_id[node] = idx
        id_to_node[idx] = node

    for u, v, d in G.edges(data=True):

        im.add_link(
            node_to_id[u],
            node_to_id[v],
            d["weight"]
        )

    im.run()

    partition = {}

    for node in im.nodes:

        partition[
            id_to_node[node.node_id]
        ] = node.module_id

    return partition

In [16]:
# ============================================================
# EVALUATION
# ============================================================

def evaluate_partition(
    G,
    partition,
    ground_truth_attr="line"
):

    nodes = list(partition.keys())

    y_true = []
    y_pred = []

    for n in nodes:

        if ground_truth_attr not in G.nodes[n]:
            continue

        y_true.append(
            G.nodes[n][ground_truth_attr]
        )

        y_pred.append(
            partition[n]
        )

    nmi = normalized_mutual_info_score(
        y_true,
        y_pred
    )

    ami = adjusted_mutual_info_score(
        y_true,
        y_pred
    )

    true_labels = pd.factorize(y_true)[0]
    pred_labels = pd.factorize(y_pred)[0]

    f1 = f1_score(
        true_labels,
        pred_labels,
        average="macro"
    )

    return nmi, ami, f1

In [17]:
# ============================================================
# MAIN
# ============================================================

results = []

for team in SEMIFINALISTS:

    print(f"\n===== {team} =====")

    G = build_team_network(team)

    print(
        f"Nodes={G.number_of_nodes()} | "
        f"Edges={G.number_of_edges()}"
    )

    # --------------------------------------------------------
    # Louvain
    # --------------------------------------------------------

    partition, Q = run_louvain(G)

    nmi, ami, f1 = evaluate_partition(
        G,
        partition
    )

    results.append({
        "team": team,
        "algorithm": "Louvain",
        "communities": len(set(partition.values())),
        "modularity_Q": round(Q, 3),
        "NMI": round(nmi, 3),
        "AMI": round(ami, 3),
        "F1_macro": round(f1, 3)
    })

    # --------------------------------------------------------
    # Leiden
    # --------------------------------------------------------

    partition, Q = run_leiden(G)

    nmi, ami, f1 = evaluate_partition(
        G,
        partition
    )

    results.append({
        "team": team,
        "algorithm": "Leiden",
        "communities": len(set(partition.values())),
        "modularity_Q": round(Q, 3),
        "NMI": round(nmi, 3),
        "AMI": round(ami, 3),
        "F1_macro": round(f1, 3)
    })

    # --------------------------------------------------------
    # Infomap
    # --------------------------------------------------------

    partition = run_infomap(G)

    nmi, ami, f1 = evaluate_partition(
        G,
        partition
    )

    results.append({
        "team": team,
        "algorithm": "Infomap",
        "communities": len(set(partition.values())),
        "modularity_Q": np.nan,
        "NMI": round(nmi, 3),
        "AMI": round(ami, 3),
        "F1_macro": round(f1, 3)
    })


===== Argentina =====


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: 

Nodes=24 | Edges=333

===== France =====


/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  true_labels = pd.factorize(y_true)[0]
/tmp/ipykernel_1715/4243135787.py:40: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pred_labels = pd.factorize(y_pred)[0]
/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  true_labels = pd.factorize(y_true)[0]
/tmp/ipykernel_1715/4243135787.py:40: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pred_labels = pd.factorize(y_pred)[0]
/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument tha

Nodes=24 | Edges=333

===== Croatia =====


/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  true_labels = pd.factorize(y_true)[0]
/tmp/ipykernel_1715/4243135787.py:40: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pred_labels = pd.factorize(y_pred)[0]
/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  true_labels = pd.factorize(y_true)[0]
/tmp/ipykernel_1715/4243135787.py:40: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pred_labels = pd.factorize(y_pred)[0]
/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument tha

Nodes=20 | Edges=281

===== Morocco =====


/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  true_labels = pd.factorize(y_true)[0]
/tmp/ipykernel_1715/4243135787.py:40: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pred_labels = pd.factorize(y_pred)[0]
/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  true_labels = pd.factorize(y_true)[0]
/tmp/ipykernel_1715/4243135787.py:40: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pred_labels = pd.factorize(y_pred)[0]
/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument tha

Nodes=25 | Edges=301


/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  true_labels = pd.factorize(y_true)[0]
/tmp/ipykernel_1715/4243135787.py:40: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pred_labels = pd.factorize(y_pred)[0]
/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  true_labels = pd.factorize(y_true)[0]
/tmp/ipykernel_1715/4243135787.py:40: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pred_labels = pd.factorize(y_pred)[0]
/tmp/ipykernel_1715/4243135787.py:39: FutureWarning: factorize with argument tha

In [18]:
# ============================================================
# RESULTS
# ============================================================

results_df = pd.DataFrame(results)

print("\n===== FINAL RESULTS =====")
print(results_df)

results_df.to_csv(
    "rq2_results.csv",
    index=False
)

print("\nResults saved to rq2_results.csv")


===== FINAL RESULTS =====
         team algorithm  communities  modularity_Q    NMI    AMI  F1_macro
0   Argentina   Louvain            4         0.097  0.086 -0.108     0.172
1   Argentina    Leiden            4         0.013  0.158 -0.015     0.169
2   Argentina   Infomap            1           NaN  0.000  0.000     0.125
3      France   Louvain            3         0.134  0.100 -0.039     0.317
4      France    Leiden            3         0.047  0.098 -0.044     0.199
5      France   Infomap            1           NaN  0.000  0.000     0.100
6     Croatia   Louvain            4         0.097  0.305  0.121     0.324
7     Croatia    Leiden            4        -0.011  0.296  0.110     0.247
8     Croatia   Infomap            1           NaN  0.000  0.000     0.100
9     Morocco   Louvain            3         0.122  0.104 -0.030     0.115
10    Morocco    Leiden            4        -0.030  0.283  0.133     0.234
11    Morocco   Infomap            1           NaN  0.000  0.000     0.10